In [7]:
pip install customtkinter

   ---------------------------------------- 0.0/296.1 kB ? eta -:--:--
   ---------------------------------------- 0.0/296.1 kB ? eta -:--:--
   ---------------------------------------- 0.0/296.1 kB ? eta -:--:--
   - -------------------------------------- 10.2/296.1 kB ? eta -:--:--
   ----- --------------------------------- 41.0/296.1 kB 393.8 kB/s eta 0:00:01
   ------------------- -------------------- 143.4/296.1 kB 1.1 MB/s eta 0:00:01
   -------------------------------------- - 286.7/296.1 kB 2.0 MB/s eta 0:00:01
   ---------------------------------------- 296.1/296.1 kB 1.5 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [1]:
import customtkinter as ctk
import random
import json
import os

# THEME SETTINGS
ctk.set_appearance_mode("dark")
ctk.set_default_color_theme("dark-blue")  # change this: blue, green, dark-blue



class QuizApp(ctk.CTk):
    def __init__(self):
        super().__init__()

        self.title("Quizzie")
        self.geometry("600x500")
        self.timer_id = None

        self.show_username_screen()
        self.quiz_data = self.load_questions()
        
        

    """
    def clear(self):
        for widget in self.winfo_children():
            widget.destroy()
    """

    def clear(self):
        if self.timer_id:
            self.after_cancel(self.timer_id)
            self.timer_id = None

        for widget in self.winfo_children():
            widget.destroy()

    # -------- USERNAME --------
    def show_username_screen(self):
        self.clear()

        ctk.CTkLabel(self, text="Enter Your Name", text_color="#FFD1DC",
                     font=("Arial", 24, "bold")).pack(pady=40)

        self.name_entry = ctk.CTkEntry(self, width=250)
        self.name_entry.pack(pady=10)

        self.error_label = ctk.CTkLabel(self, text="", text_color="red")
        self.error_label.pack()

        ctk.CTkButton(self, text="Start",
                      command=self.save_name).pack(pady=20)

    def save_name(self):
        self.username = self.name_entry.get().strip()

        if not self.username:
            self.error_label.configure(text="Name cannot be empty")
            return

        self.show_start()

    # -------- START --------
    def show_start(self):
        self.clear()

        ctk.CTkLabel(self, text=f"Welcome, {self.username}", text_color="#FFD1DC",
                     font=("Arial", 20, "bold")).pack(pady=20)

        for level in ["easy", "medium", "hard"]:
            ctk.CTkButton(self,
                          text=level.capitalize(),
                          width=200,
                          command=lambda l=level: self.start_quiz(l)
                          ).pack(pady=8)

        ctk.CTkButton(self, text="View My History",
                      command=self.show_history).pack(pady=20)

    def load_questions(self):
        try:
            with open("questions.json", "r") as f:
                return json.load(f)
        except:
            return {}

    # -------- QUIZ --------
    def start_quiz(self, level):
        self.level = level
        self.score = 0
        self.q_index = 0
        self.questions = self.quiz_data.get(level, [])
        random.shuffle(self.questions)
        self.show_question()

    def show_question(self):
        self.clear()

        if self.q_index >= len(self.questions):
            self.show_result()
            return

        self.time_left = 10
        data = self.questions[self.q_index]
        q = data["question"]

        ctk.CTkLabel(self, text_color="#FFD1DC",
                     text=f"Q {self.q_index+1}/{len(self.questions)}"
                     ).pack(anchor="ne", padx=10)

        self.timer_label = ctk.CTkLabel(self,
                                        text=f"Time: {self.time_left}",
                                        text_color="red")
        self.timer_label.pack(anchor="ne", padx=10)

        ctk.CTkLabel(self,
                     text=q, text_color="#FFD1DC",
                     wraplength=500,
                     font=("Arial", 18)).pack(pady=20)

        self.selected = ctk.StringVar()

        for opt in data["options"]:
            ctk.CTkRadioButton(self,
                               text=opt,
                               variable=self.selected,
                               value=opt).pack(anchor="w", padx=40)

        self.feedback = ctk.CTkLabel(self, text="")
        self.feedback.pack(pady=10)

        ctk.CTkButton(self, text="Submit",
                      command=self.check_answer).pack(pady=15)

        self.update_timer()

    """
    def update_timer(self):
        self.timer_label.configure(text=f"Time: {self.time_left}")
        if self.time_left > 0:
            self.time_left -= 1
            self.after(1000, self.update_timer)
        else:
            self.feedback.configure(text="Time's up!")
            self.after(1000, self.next_q) 
    """

    def update_timer(self):
        self.timer_label.configure(text=f"Time: {self.time_left}")

        if self.time_left > 0:
            self.time_left -= 1
            self.timer_id = self.after(1000, self.update_timer)
        else:
            self.feedback.configure(text="Time's up!")
            self.timer_id = self.after(1000, self.next_q)

    def check_answer(self):
        data = self.questions[self.q_index]
        q = data["question"]

        import random

        correct_msgs = [
                    "Big brain moment!",
                    "You’re too smart for this!",
                    "Easy win!",
                    "💯 Nailed it!"
                ]
                
        wrong_msgs = [
                    "Oops... not quite",
                    "Try again, genius",
                    "That hurt your reputation"
                ]
    
        if self.selected.get() == data["answer"]:
            self.score += 1
            self.feedback.configure(text=random.choice(correct_msgs), text_color="green")
        else:
            self.feedback.configure(text=f"{random.choice(wrong_msgs)}\nAnswer: {data['answer']}",
                                   text_color="red"
                                   )
        

        self.after(1000, self.next_q)

    def next_q(self):
        self.q_index += 1
        self.show_question()

    # -------- SAVE --------
    def save_score(self):
        entry = {
            "user": self.username,
            "difficulty": self.level,
            "score": self.score,
            "total": len(self.questions)
        }

        if os.path.exists("scores.json"):
            try:
                with open("scores.json", "r") as f:
                    data = json.load(f)
            except:
                data = []
        else:
            data = []

        data.append(entry)

        with open("scores.json", "w") as f:
            json.dump(data, f, indent=4)

    # -------- RESULT --------
    def show_result(self):
        self.clear()
        self.save_score()

        ctk.CTkLabel(self, text=random.choice([
                                                    "Brain Battle Over!"
                                                ]),
                     font=("Arial", 24, "bold")).pack(pady=30)

        ctk.CTkLabel(self,
                     text=f"{self.username}, Score: {self.score}/{len(self.questions)}"
                     ).pack(pady=10)

        ctk.CTkButton(self, text="Play Again",
                      command=self.show_start).pack(pady=10)

    # -------- HISTORY (USER ONLY + BEST SCORE) --------
    def show_history(self):
        self.clear()

        ctk.CTkLabel(self, text="My History",
                     font=("Arial", 24, "bold")).pack(pady=20)

        scroll = ctk.CTkScrollableFrame(self, height=300)
        scroll.pack(fill="both", expand=True, padx=20)

        if os.path.exists("scores.json"):
            with open("scores.json", "r") as f:
                try:
                    data = json.load(f)
                except:
                    data = []
        else:
            data = []

        user_data = [d for d in data if d.get("user") == self.username]

        if not user_data:
            ctk.CTkLabel(scroll, text="No history").pack()
        else:
            best = max(user_data, key=lambda x: x["score"])

            ctk.CTkLabel(scroll,
                         text=f"🏆 Best Score: {best['score']}/{best['total']}",
                         text_color="green").pack(pady=10)

            for i, entry in enumerate(reversed(user_data), 1):
                text = f"{i}. {entry['difficulty']} → {entry['score']}/{entry['total']}"
                ctk.CTkLabel(scroll, text=text).pack(anchor="w", pady=2)

        ctk.CTkButton(self, text="Back",
                      command=self.show_start).pack(pady=15)


app = QuizApp()
app.mainloop()